# Churn en telecomunicaciones

Trabajo Final Integrador. Clasificación retrospectiva y campaña con cupo del 20%.

**Autores:** Ana Espinosa, Micaela Venezia, Gonzalo Lavecchia, Nahuel Jimenez y Emiliano Ezequiel Greco Vairo.

El informe se apoya en la ejecución local del experimento adjunto. Este notebook permite repetirla en Colab. No existe un horizonte temporal verificable de predicción en el dataset.

## 1 Instalar dependencias

En Colab, si numpy o pandas ya estaban cargados, reiniciar el entorno tras esta celda y continuar desde la siguiente. No aplicar SMOTE antes de separar entrenamiento y prueba.

In [ ]:
%pip install numpy==2.3.5 pandas==3.0.1 scipy==1.16.2 scikit-learn==1.7.2 imbalanced-learn==0.14.0 xgboost==3.1.1 joblib==1.5.2 threadpoolctl==3.6.0


## 2 Guardar el experimento reproducible

El código incluido realiza descarga con checksum, separación estratificada, EDA sobre train, preparación dentro de pipelines y CV de cinco folds. Compara LR, RF y XGBoost, ponderación, SMOTENC y ablación de TotalCharges. La decisión y el umbral se fijan antes de evaluar el holdout.

In [ ]:
from pathlib import Path
import json
carpeta = Path("/content/proyecto_churn") if Path("/content").exists() else Path.cwd() / "proyecto_churn"
carpeta.mkdir(exist_ok=True)
script = '"""Experimento reproducible. No usar el holdout para elegir modelos ni umbrales."""\nfrom pathlib import Path\nimport sys\nif (Path(__file__).parent / \'ml_direct\').exists():\n    sys.path.insert(0, str(Path(__file__).parent / \'ml_direct\'))\nimport hashlib\nimport json\nimport platform\nimport time\nimport urllib.request\nimport numpy as np\nimport pandas as pd\nimport sklearn\nimport imblearn\nimport xgboost\nfrom sklearn.base import clone\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,\n    recall_score, f1_score, accuracy_score, confusion_matrix, brier_score_loss)\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler\nfrom imblearn.over_sampling import SMOTENC\nfrom imblearn.pipeline import Pipeline as ImbPipeline\nfrom xgboost import XGBClassifier\n\nSEED = 42\nURL = \'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv\'\nSHA256 = \'16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91\'\nROOT = Path(__file__).resolve().parent\nOUT = ROOT / \'resultados\'\nOUT.mkdir(exist_ok=True)\n\ndef top_metrics(y, p):\n    y, p = np.asarray(y), np.asarray(p)\n    k = int(np.ceil(.2 * len(y)))\n    # Orden estable. El orden inicial del archivo resuelve empates sin consultar y.\n    ix = np.argsort(-p, kind=\'stable\')[:k]\n    hits = int(y[ix].sum())\n    return {\'p20\': hits/k, \'r20\': hits/int(y.sum()), \'lift20\': (hits/k)/y.mean(),\n            \'k\': k, \'hits\': hits}\n\ndef metrics(y, p, threshold=.5):\n    pred = np.asarray(p) >= threshold\n    return {\'roc_auc\':roc_auc_score(y,p), \'ap\':average_precision_score(y,p),\n            \'precision\':precision_score(y,pred,zero_division=0),\n            \'recall\':recall_score(y,pred), \'f1\':f1_score(y,pred),\n            \'accuracy\':accuracy_score(y,pred), \'brier\':brier_score_loss(y,p),\n            **top_metrics(y,p)}\n\ndef prepare(frame):\n    x = frame.drop(columns=[\'customerID\',\'Churn\',\'gender\',\'SeniorCitizen\']).copy()\n    x[\'TotalCharges\'] = pd.to_numeric(x[\'TotalCharges\'],errors=\'coerce\')\n    x[\'total_sin_facturar\'] = x[\'TotalCharges\'].isna().map({True:\'Yes\',False:\'No\'})\n    x.loc[x[\'TotalCharges\'].isna() & x[\'tenure\'].eq(0),\'TotalCharges\'] = 0.\n    services=[\'PhoneService\',\'MultipleLines\',\'OnlineSecurity\',\'OnlineBackup\',\n              \'DeviceProtection\',\'TechSupport\',\'StreamingTV\',\'StreamingMovies\']\n    x[\'cantidad_servicios\'] = frame[services].eq(\'Yes\').sum(axis=1)\n    x[\'pago_automatico\'] = frame[\'PaymentMethod\'].str.contains(\'automatic\').map({True:\'Yes\',False:\'No\'})\n    return x\n\ndef preprocessor(x):\n    num = x.select_dtypes(include=\'number\').columns.tolist()\n    cat = [c for c in x if c not in num]\n    return ColumnTransformer([\n        (\'num\',Pipeline([(\'impute\',SimpleImputer(strategy=\'median\')),\n                         (\'scale\',StandardScaler())]),num),\n        (\'cat\',Pipeline([(\'impute\',SimpleImputer(strategy=\'most_frequent\')),\n                         (\'ohe\',OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False))]),cat)\n    ])\n\ndef smotenc_pipeline(x):\n    num=x.select_dtypes(include=\'number\').columns.tolist()\n    cat=[c for c in x if c not in num]\n    before=ColumnTransformer([\n        (\'num\',Pipeline([(\'impute\',SimpleImputer(strategy=\'median\')),(\'scale\',StandardScaler())]),num),\n        (\'cat\',Pipeline([(\'impute\',SimpleImputer(strategy=\'most_frequent\')),\n                         (\'ordinal\',OrdinalEncoder(handle_unknown=\'use_encoded_value\',unknown_value=-1))]),cat)])\n    cat_ix=list(range(len(num),len(num)+len(cat)))\n    after=ColumnTransformer([(\'num\',\'passthrough\',list(range(len(num)))),\n                            (\'cat\',OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False),cat_ix)])\n    return ImbPipeline([(\'before\',before),(\'smotenc\',SMOTENC(categorical_features=cat_ix,random_state=SEED)),\n                        (\'after\',after),(\'model\',LogisticRegression(C=1,max_iter=2000,random_state=SEED))])\n\ndef run():\n    data=ROOT/\'Telco-Customer-Churn.csv\'\n    if not data.exists():\n        data.write_bytes(urllib.request.urlopen(URL).read())\n    assert hashlib.sha256(data.read_bytes()).hexdigest()==SHA256, \'Cambió el archivo de datos\'\n    raw=pd.read_csv(data)\n    assert len(raw)==7043 and raw.shape[1]==21\n    assert raw[\'customerID\'].is_unique\n    y=raw[\'Churn\'].eq(\'Yes\').astype(int)\n    idx_train,idx_test=train_test_split(np.arange(len(raw)),test_size=.2,stratify=y,random_state=SEED)\n    # Se persisten índices antes del análisis y de las búsquedas.\n    pd.DataFrame({\'row\':np.r_[idx_train,idx_test],\n                  \'split\':[\'train\']*len(idx_train)+[\'test\']*len(idx_test)}).to_csv(OUT/\'particion.csv\',index=False)\n    X=prepare(raw)\n    Xtr,Xte=X.iloc[idx_train].reset_index(drop=True),X.iloc[idx_test].reset_index(drop=True)\n    ytr,yte=y.iloc[idx_train].reset_index(drop=True),y.iloc[idx_test].reset_index(drop=True)\n    trainraw=raw.iloc[idx_train].copy()\n    trainraw[\'target\']=y.iloc[idx_train]\n    eda={}\n    for field in [\'Contract\',\'InternetService\',\'PaymentMethod\']:\n        eda[field]=trainraw.groupby(field)[\'target\'].agg([\'count\',\'sum\',\'mean\']).reset_index().to_dict(\'records\')\n    trainraw[\'antiguedad\']=pd.cut(trainraw[\'tenure\'],[-1,12,24,48,72],labels=[\'0 a 12\',\'13 a 24\',\'25 a 48\',\'49 a 72\'])\n    eda[\'tenure\']=trainraw.groupby(\'antiguedad\',observed=True)[\'target\'].agg([\'count\',\'sum\',\'mean\']).reset_index().to_dict(\'records\')\n    nt=pd.to_numeric(trainraw[\'TotalCharges\'],errors=\'coerce\')\n    eda[\'corr_total_tenure\']=float(nt.corr(trainraw[\'tenure\']))\n    eda[\'corr_total_monthly\']=float(nt.corr(trainraw[\'MonthlyCharges\']))\n    eda[\'ranges\']={c:[float(pd.to_numeric(trainraw[c],errors=\'coerce\').min()),float(pd.to_numeric(trainraw[c],errors=\'coerce\').max())] for c in [\'tenure\',\'MonthlyCharges\',\'TotalCharges\']}\n    eda[\'median_monthly_by_churn\']=trainraw.groupby(\'Churn\')[\'MonthlyCharges\'].median().to_dict()\n    blank=pd.to_numeric(raw[\'TotalCharges\'],errors=\'coerce\').isna()\n    quality={\'n\':len(raw),\'columns\':raw.shape[1],\'positive\':int(y.sum()),\'prevalence\':float(y.mean()),\n             \'missing_total\':int(blank.sum()),\'missing_total_tenure0\':int((blank & raw[\'tenure\'].eq(0)).sum()),\n             \'duplicate_ids\':int(raw[\'customerID\'].duplicated().sum()),\'duplicate_rows\':int(raw.duplicated().sum()),\n             \'other_missing\':int(raw.drop(columns=\'TotalCharges\').isna().sum().sum()),\n             \'n_train\':len(ytr),\'n_test\':len(yte),\'train_positive\':int(ytr.sum()),\'test_positive\':int(yte.sum()),\n             \'features\':X.columns.tolist()}\n    specs=[]\n    for c in [.1,1,10]:\n        specs.append((f\'LR C={c:g}\', \'LR\', Pipeline([(\'pre\',preprocessor(Xtr)),(\'model\',LogisticRegression(C=c,max_iter=2000,random_state=SEED))]),False))\n    specs.append((\'LR balanced\',\'LR\',Pipeline([(\'pre\',preprocessor(Xtr)),(\'model\',LogisticRegression(C=1,class_weight=\'balanced\',max_iter=2000,random_state=SEED))]),False))\n    for leaf in [3,10]:\n        specs.append((f\'RF hoja={leaf}\',\'RF\',Pipeline([(\'pre\',preprocessor(Xtr)),(\'model\',RandomForestClassifier(n_estimators=250,min_samples_leaf=leaf,max_features=\'sqrt\',random_state=SEED,n_jobs=2))]),False))\n    for depth in [2,3]:\n        specs.append((f\'XGB profundidad={depth}\',\'XGB\',Pipeline([(\'pre\',preprocessor(Xtr)),(\'model\',XGBClassifier(n_estimators=200,max_depth=depth,learning_rate=.04,subsample=.8,colsample_bytree=.8,reg_lambda=1,random_state=SEED,n_jobs=2,eval_metric=\'logloss\'))]),False))\n    specs.append((\'LR SMOTENC\',\'SMOTENC\',smotenc_pipeline(Xtr),False))\n    ab=Xtr.drop(columns=\'TotalCharges\')\n    specs.append((\'LR sin TotalCharges\',\'Ablacion\',Pipeline([(\'pre\',preprocessor(ab)),(\'model\',LogisticRegression(C=1,max_iter=2000,random_state=SEED))]),True))\n    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)\n    allrows=[]; oofs={}; foldrows=[]\n    for name,family,est,drop_total in specs:\n        t=time.time(); xx=Xtr.drop(columns=\'TotalCharges\') if drop_total else Xtr\n        oof=np.zeros(len(ytr)); ms=[]\n        for fold,(a,b) in enumerate(cv.split(xx,ytr),1):\n            model=clone(est).fit(xx.iloc[a],ytr.iloc[a])\n            p=model.predict_proba(xx.iloc[b])[:,1]; oof[b]=p\n            m=metrics(ytr.iloc[b],p); ms.append(m)\n            foldrows.append({\'name\':name,\'fold\':fold,**m})\n        row={\'name\':name,\'family\':family}\n        for key in [\'p20\',\'r20\',\'roc_auc\',\'ap\',\'precision\',\'recall\',\'f1\',\'accuracy\']:\n            row[key]=float(np.mean([m[key] for m in ms])); row[key+\'_sd\']=float(np.std([m[key] for m in ms],ddof=1))\n        allrows.append(row); oofs[name]=oof\n        print(name, \'P20\',round(row[\'p20\'],4),\'AP\',round(row[\'ap\'],4),\'seg\',round(time.time()-t,1),flush=True)\n    rows=pd.DataFrame(allrows)\n    eligible=rows[rows.family.ne(\'Ablacion\')]\n    bestp=eligible.p20.max()\n    # Regla previa: dentro de 1 pp del mejor P20, preferir LR sin remuestreo.\n    close=eligible[eligible.p20>=bestp-.01]\n    simple=close[close[\'name\'].isin([\'LR C=0.1\',\'LR C=1\',\'LR C=10\'])]\n    selected=(simple if len(simple) else close).sort_values([\'p20\',\'ap\'],ascending=False).iloc[0]\n    name=selected[\'name\']; p_oof=oofs[name]\n    choices=[]\n    for threshold in np.linspace(.05,.8,151):\n        pred=p_oof>=threshold\n        pr=precision_score(ytr,pred,zero_division=0); re=recall_score(ytr,pred)\n        if pr>=.5 and re>=.75:\n            choices.append((f1_score(ytr,pred),float(threshold),pr,re))\n    chosen_threshold=max(choices)[1] if choices else .5\n    # Decisión registrada ANTES de predecir/evaluar el holdout.\n    decision={\'selected\':name,\'selection_rule\':\'P20 medio de CV; tolerancia 1 pp para LR simple; desempate AP\',\n              \'threshold_oof\':chosen_threshold,\'threshold_target_feasible_oof\':bool(choices)}\n    (OUT/\'decision_previa.json\').write_text(json.dumps(decision,indent=2),encoding=\'utf-8\')\n    spec=next(s for s in specs if s[0]==name)\n    final=clone(spec[2]).fit(Xtr,ytr)\n    p=final.predict_proba(Xte)[:,1]\n    test_default=metrics(yte,p)\n    test_threshold=metrics(yte,p,chosen_threshold)\n    k=int(np.ceil(.2*len(yte))); ix=np.argsort(-p,kind=\'stable\')[:k]\n    campaign=np.zeros(len(yte),dtype=int); campaign[ix]=1\n    test_default[\'confusion_05\']=confusion_matrix(yte,p>=.5).tolist()\n    test_threshold[\'confusion\']=confusion_matrix(yte,p>=chosen_threshold).tolist()\n    campaign_cm=confusion_matrix(yte,campaign).tolist()\n    rng=np.random.default_rng(SEED)\n    boots=[]\n    for _ in range(2000):\n        b=rng.integers(0,len(yte),len(yte))\n        boots.append(top_metrics(yte.iloc[b],p[b])[\'p20\'])\n    ci=np.quantile(boots,[.025,.975]).tolist()\n    audits={}\n    for col in [\'gender\',\'SeniorCitizen\']:\n        fr=raw.iloc[idx_test][[col]].reset_index(drop=True)\n        fr[\'y\']=yte; fr[\'score\']=p; fr[\'contact\']=campaign\n        audits[col]=[]\n        for group,g in fr.groupby(col):\n            hits=int((g.y*g.contact).sum()); contacts=int(g.contact.sum())\n            audits[col].append({\'grupo\':str(group),\'n\':len(g),\'positivos\':int(g.y.sum()),\n                               \'contactos\':contacts,\'seleccion\':float(g.contact.mean()),\n                               \'p20_grupo\':hits/contacts if contacts else None,\n                               \'recall_grupo\':hits/int(g.y.sum()) if g.y.sum() else None})\n    # Coeficientes descriptivos del modelo elegido si es regresión logística.\n    coefficients=[]\n    if name.startswith(\'LR C=\'):\n        features=final.named_steps[\'pre\'].get_feature_names_out()\n        co=final.named_steps[\'model\'].coef_[0]\n        coefficients=pd.DataFrame({\'feature\':features,\'coefficient\':co}).sort_values(\'coefficient\').to_dict(\'records\')\n    pd.DataFrame({\'row\':idx_test,\'y_true\':yte,\'score\':p,\'top20\':campaign}).to_csv(OUT/\'predicciones_test.csv\',index=False)\n    rows.to_csv(OUT/\'comparacion_cv.csv\',index=False)\n    pd.DataFrame(foldrows).to_csv(OUT/\'metricas_folds.csv\',index=False)\n    pd.DataFrame({\'row\':idx_train,\'y_true\':ytr,\'oof_selected\':p_oof}).to_csv(OUT/\'predicciones_oof.csv\',index=False)\n    result={\'quality\':quality,\'eda_train\':eda,\'cv\':allrows,\'decision\':decision,\n            \'test_05\':test_default,\'test_threshold\':test_threshold,\'campaign_confusion\':campaign_cm,\n            \'p20_bootstrap95\':ci,\'audit\':audits,\'coefficients\':coefficients,\n            \'versions\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\n                        \'scikit-learn\':sklearn.__version__,\'imbalanced-learn\':imblearn.__version__,\'xgboost\':xgboost.__version__},\n            \'url\':URL,\'sha256\':SHA256}\n    (OUT/\'resultados.json\').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    print(json.dumps({\'decision\':decision,\'test\':test_default,\'threshold\':test_threshold,\'ci\':ci},indent=2),flush=True)\n    return result\n\nif __name__==\'__main__\':\n    run()\n'
(carpeta / "analisis_churn.py").write_text(script, encoding="utf-8")


## 3 Ejecutar el análisis

Se preserva la distribución original en validación y test. Solo SMOTENC remuestrea la porción de entrenamiento de cada fold. El script imprime resultados por configuración y la evaluación del modelo elegido.

In [ ]:
import subprocess, sys
proceso = subprocess.run([sys.executable, str(carpeta / "analisis_churn.py")], capture_output=True, text=True)
print(proceso.stdout)
if proceso.returncode:
    print(proceso.stderr)
    raise RuntimeError("Falló el experimento")


## 4 Revisar los resultados

La clase positiva es Churn = Yes. P20 mide el acierto dentro del cupo y R20 su cobertura. El umbral auxiliar de clasificación no define la lista de contactos.

In [ ]:
import pandas as pd
r = json.loads((carpeta / "resultados" / "resultados.json").read_text(encoding="utf-8"))
display(pd.DataFrame(r["cv"])[["name", "p20", "p20_sd", "roc_auc", "ap", "precision", "recall", "f1"]])
print("Decisión:", r["decision"])
print("Prueba a 0.50:", r["test_05"])
print("Prueba al umbral auxiliar:", r["test_threshold"])
print("IC bootstrap P20:", r["p20_bootstrap95"])


## 5 Auditoría y escenario económico

Los atributos gender y SeniorCitizen no intervienen como predictores. Revisar diferencias de selección y error. La exclusión de atributos sensibles no elimina automáticamente posibles sesgos.

In [ ]:
for atributo, filas in r["audit"].items():
    print(atributo)
    display(pd.DataFrame(filas))

n = r["quality"]["n_test"]
positivos = r["quality"]["test_positive"]
k = r["test_05"]["k"]
aciertos = r["test_05"]["hits"]
# Supuestos ilustrativos, no ingresos ni efectos causales observados.
costo_contacto, margen_retenido, eficacia = 5, 120, 0.15
azar = k * positivos / n
neto_modelo = aciertos * eficacia * margen_retenido - k * costo_contacto
neto_azar = azar * eficacia * margen_retenido - k * costo_contacto
print({"casos_azar_esperados": azar, "beneficio_modelo": neto_modelo, "beneficio_azar": neto_azar, "incremental": neto_modelo - neto_azar})


## 6 Exportar evidencias y completar publicación

Guardar este notebook en Drive y publicar el código en el repositorio público del equipo. Agregar los enlaces reales al envío académico. El ahorro y la reducción del churn deben medirse en un piloto aleatorizado con datos temporales.

In [ ]:
import shutil
archivo = shutil.make_archive(str(carpeta / "evidencias_churn"), "zip", carpeta / "resultados")
print("Evidencias:", archivo)
try:
    from google.colab import files
    files.download(archivo)
except ImportError:
    pass
